In [1]:
from argparse import Namespace

In [ ]:
from tqdm import tqdm

In [2]:
import pandas as pd

In [3]:
#from eap.graph import Graph
from eap import graph

In [4]:
from eap.utils import tokenize_plus

In [5]:
from transformer_lens import HookedTransformer

In [6]:
from ablation_utils.utils import make_hooks

In [ ]:
import importlib

In [ ]:
importlib.reload(graph)

<module 'eap.graph' from '/mounts/work/sgerstner/RW_functionalities/EAP-IG/src/eap/graph.py'>

In [8]:
DEVICE='cuda:1'

In [9]:
my_graph = graph.Graph.from_pt('../RW_functionalities_results/full_graph_with_subnodes.pt')

In [10]:
#subnodes_df = my_graph.subnodes_to_pandas('../RW_functionalities_results/subnodes.csv')
subnodes_df = pd.read_csv('../RW_functionalities_results/subnodes.csv')

In [11]:
subnodes_df.head(40)

,Unnamed: 0,name,layer,node_name,subnode_index,score
0,0,input,0,input,NaN,0.000000e+00
1,1,a0.h0,0,a0.h0,NaN,0.000000e+00
2,2,a0.h1,0,a0.h1,NaN,0.000000e+00
3,3,a0.h2,0,a0.h2,NaN,0.000000e+00
4,4,a0.h3,0,a0.h3,NaN,0.000000e+00
5,5,a0.h4,0,a0.h4,NaN,0.000000e+00
6,6,a0.h5,0,a0.h5,NaN,0.000000e+00
7,7,a0.h6,0,a0.h6,NaN,0.000000e+00
8,8,a0.h7,0,a0.h7,NaN,0.000000e+00
9,9,a0.h8,0,a0.h8,NaN,0.000000e+00


In [12]:
sorted_df = subnodes_df.sort_values("score", ascending=False).head(49)
print(sorted_df)

      Unnamed: 0      name  layer node_name  subnode_index     score
1549        1549  m31.8581     31       m31         8581.0  1.510321
1405        1405  m30.9996     30       m30         9996.0  1.103545
527          527       m12     12       m12            NaN  0.726136
1424        1424   a31.h12     31   a31.h12            NaN  0.663149
1339        1339   a30.h10     30   a30.h10            NaN  0.620403
1320        1320  m29.7261     29       m29         7261.0  0.571579
685          685       m16     16       m16            NaN  0.542899
1444        1444       m31     31       m31            NaN  0.520416
981          981       m23     23       m23            NaN  0.467276
1363        1363   m30.732     30       m30          732.0  0.456894
1345        1345   a30.h16     30   a30.h16            NaN  0.377761
1043        1043  m24.7935     24       m24         7935.0  0.374789
1306        1306  m29.4131     29       m29         4131.0  0.373894
566          566       m13     13 

In [25]:
sorted_filtered_df = subnodes_df[subnodes_df["subnode_index"].notna() & (subnodes_df["score"]>0)].sort_values("score", ascending=False)
print(sorted_filtered_df.shape)

(246, 6)


In [26]:
print(sorted_filtered_df.head(40))

      Unnamed: 0       name  layer node_name  subnode_index     score
1549        1549   m31.8581     31       m31         8581.0  1.510321
1405        1405   m30.9996     30       m30         9996.0  1.103545
1320        1320   m29.7261     29       m29         7261.0  0.571579
1363        1363    m30.732     30       m30          732.0  0.456894
1043        1043   m24.7935     24       m24         7935.0  0.374789
1306        1306   m29.4131     29       m29         4131.0  0.373894
1489        1489   m31.3498     31       m31         3498.0  0.193198
1377        1377   m30.3705     30       m30         3705.0  0.114631
1506        1506   m31.4568     31       m31         4568.0  0.114614
1148        1148   m26.5558     26       m26         5558.0  0.085032
612          612   m14.4516     14       m14         4516.0  0.066492
1253        1253   m28.6703     28       m28         6703.0  0.039890
1524        1524   m31.5973     31       m31         5973.0  0.037797
1551        1551   m

In [27]:
print(sorted_filtered_df.tail(40))

      Unnamed: 0       name  layer node_name  subnode_index     score
382          382      m8.22      8        m8           22.0  0.000156
861          861   m20.1821     20       m20         1821.0  0.000153
208          208     m4.382      4        m4          382.0  0.000143
260          260    m5.6293      5        m5         6293.0  0.000141
1481        1481   m31.2938     31       m31         2938.0  0.000140
1519        1519   m31.5751     31       m31         5751.0  0.000125
1252        1252   m28.6431     28       m28         6431.0  0.000120
129          129    m2.3729      2        m2         3729.0  0.000117
1326        1326  m29.10283     29       m29        10283.0  0.000110
1567        1567  m31.10439     31       m31        10439.0  0.000108
128          128     m2.211      2        m2          211.0  0.000108
1080        1080    m25.782     25       m25          782.0  0.000107
1309        1309   m29.5237     29       m29         5237.0  0.000105
689          689  m1

## Evaluating via ablations

In [37]:
model = HookedTransformer.from_pretrained_no_processing(#with predefined neurons, our ablation code is implementation-invariant
        'allenai/OLMo-7B-0424-hf',
        device=DEVICE,
)

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Loaded pretrained model allenai/OLMo-7B-0424-hf into HookedTransformer


In [15]:
sequence = "Yesterday (21 December) the Government announced a package of support for hospitality and leisure businesses that are losing trade because of the O"

In [16]:
tokens, attention_mask, input_lengths, n_pos = tokenize_plus(model, [sequence])

In [17]:
args = Namespace(
    gate='-', post='+',
    activation_location='mlp.hook_post',
    intervention_type='zero_ablation',
)

In [28]:
logits = model(tokens, attention_mask=attention_mask)
score = logits[...,-1,model.to_single_token('mic')].item()
score_list = [score]
print(score)

21.09280776977539


In [29]:
hook_list = []
for i,row in sorted_filtered_df.iterrows():
    hook_list.extend(
        make_hooks(
            args,
            layer=row["layer"], neuron=int(row["subnode_index"]),
            mean_value=0.0,
        )
    )
    with model.hooks(hook_list):
        logits = model(tokens, attention_mask=attention_mask)
        score = logits[...,-1,model.to_single_token('mic')].item()
        score_list.append(score)
        print(len(score_list)-1, ', ', score)

1 ,  21.100914001464844
2 ,  19.696537017822266
3 ,  18.707660675048828
4 ,  17.817819595336914
5 ,  16.941993713378906
6 ,  15.728745460510254
7 ,  15.728745460510254
8 ,  15.55099105834961
9 ,  15.55099105834961
10 ,  14.758173942565918
11 ,  14.822172164916992
12 ,  12.082956314086914
13 ,  12.082956314086914
14 ,  12.070569038391113
15 ,  9.495447158813477
16 ,  9.495447158813477
17 ,  9.234901428222656
18 ,  9.234901428222656
19 ,  9.208263397216797
20 ,  9.208263397216797
21 ,  9.208263397216797
22 ,  9.208263397216797
23 ,  9.208263397216797
24 ,  9.213817596435547
25 ,  9.215895652770996
26 ,  9.215895652770996
27 ,  9.215895652770996
28 ,  9.215895652770996
29 ,  9.20640754699707
30 ,  9.214571952819824
31 ,  9.210451126098633
32 ,  9.20858383178711
33 ,  9.20858383178711
34 ,  9.208840370178223
35 ,  9.2221097946167
36 ,  9.2221097946167
37 ,  9.220149993896484
38 ,  9.220149993896484
39 ,  9.220155715942383
40 ,  9.220155715942383
41 ,  9.195068359375
42 ,  9.192611694335938

## I'm confused about the number of weakening neurons...

In [30]:
from weight_analysis_utils.utils import cos

In [33]:
weakening_df = subnodes_df[subnodes_df["subnode_index"].notna()].sort_values("score", ascending=False)

In [49]:
len(weakening_df)

526

In [65]:
from numpy import sign

In [ ]:
for i,row in weakening_df.iterrows():
    layer=row["layer"]
    neuron=int(row["subnode_index"])
    inout = cos(model.blocks[layer].mlp.W_in[:,neuron], model.blocks[layer].mlp.W_out[neuron,:]).item()
    gateout = cos(model.blocks[layer].mlp.W_gate[:,neuron], model.blocks[layer].mlp.W_out[neuron,:]).abs().item()
    gatein = cos(model.blocks[layer].mlp.W_gate[:,neuron], model.blocks[layer].mlp.W_in[:,neuron]).abs().item()
    if inout>=-.5:
        print(layer, neuron, 'inout:', inout)
    if gateout<=.5:
        print(layer, neuron, 'abs gateout:', gateout)
    if gatein*sign(gateout)<=.5:
        print(layer, neuron, 'abs gatein:', gatein)

In [42]:
import gc
from torch.cuda import empty_cache

In [45]:
import os

In [43]:
gc.collect()
empty_cache()

In [46]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"

In [47]:
model.process_weights_()

In [67]:
for i,row in weakening_df.iterrows():
    layer=row["layer"]
    neuron=int(row["subnode_index"])
    inout = cos(model.blocks[layer].mlp.W_in[:,neuron], model.blocks[layer].mlp.W_out[neuron,:]).item()
    gateout = cos(model.blocks[layer].mlp.W_gate[:,neuron], model.blocks[layer].mlp.W_out[neuron,:]).item()
    gatein = cos(model.blocks[layer].mlp.W_gate[:,neuron], model.blocks[layer].mlp.W_in[:,neuron]).item()
    if inout>=-.5:
        print(layer, neuron, 'inout:', inout)
    if abs(gateout)<=.5:
        print(layer, neuron, 'abs gateout:', gateout)
    if gatein*sign(gateout)>=-.5:
        print(layer, neuron, 'resigned gatein:', gatein)

In [52]:
from torch import load

In [56]:
data = load('../RW_functionalities_results/results/allenai/OLMo-7B-0424-hf/refactored/data.pt')

In [57]:
data.keys()

dict_keys(['d_model', 'linout', 'gatelin', 'gateout', 'beta', 'randomness', 'norm_gate', 'norm_in', 'norm_out', 'norm_in_out', 'categories', 'category_stats', 'W_gate_start_to_end', 'W_in_start_to_end', 'W_out_start_to_end'])

In [58]:
data['category_stats']

{(1,
  1,
  1): tensor([ 1,  1,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  2,  1, 15, 11,  1, 11,  1,  1,  1]),
 (1,
  1,
  0): tensor([ 0,  2,  6, 10, 14, 38, 18, 16,  9,  1,  4,  1,  2,  2, 13,  0,  1,  4,
          2,  6,  3,  5,  6,  8,  6,  6, 18,  2, 10,  0,  0,  0]),
 (1,
  0,
  1): tensor([1397, 2867, 7379, 6966, 6223, 7206, 7480, 7661, 7076, 6445, 5526, 4821,
         4279, 3926, 3876, 3768, 3420, 3312, 3488, 3692, 3030, 3305, 3075, 3090,
         2342, 2515, 1719, 1066,  404,   50,   39,  228]),
 (1,
  0,
  0): tensor([619,   3,   6,   1,   0,   2,   1,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   1,   1,   0,   0,   0,   2,   4,   7,  10,   1,
           4,   1,   7,   6]),
 (0,
  1,
  1): tensor([  34,  120,  270,  497,  601, 1022, 1156,  921,  839,  893,  710,  697,
          642,  685,  887,  923,  773,  636,  687,  866, 1155, 1162, 1215, 1183,
         1012, 1107,  997,  793,  504,   59,   92,  676])

In [60]:
data['category_stats'][(-1,1,1)]

tensor([  5,  23,   8,   5,  11,  19,   7,   6,   2,   3,   2,   7,   6,   9,
          8,   3,   4,  11,  11,  12,  11,   9,   7,  17,  15,  19,  29,  14,
         19,  36,  50, 138])

In [62]:
for key,value in data['category_stats'].items():
    print(key, value.sum())

(1, 1, 1) tensor(47)
(1, 1, 0) tensor(213)
(1, 0, 1) tensor(121671)
(1, 0, 0) tensor(676)
(0, 1, 1) tensor(23814)
(0, 1, 0) tensor(128)
(0, 0, 1) tensor(192376)
(-1, 1, 1) tensor(526)
(-1, 1, 0) tensor(321)
(-1, 0, 1) tensor(12440)
(-1, 0, 0) tensor(44)


In [68]:
model.cfg.normalization_type

'LNPre'

In [69]:
model.state_dict().keys()

odict_keys(['embed.W_E', 'blocks.0.attn.W_Q', 'blocks.0.attn.W_O', 'blocks.0.attn.b_Q', 'blocks.0.attn.b_O', 'blocks.0.attn.W_K', 'blocks.0.attn.W_V', 'blocks.0.attn.b_K', 'blocks.0.attn.b_V', 'blocks.0.attn.mask', 'blocks.0.attn.IGNORE', 'blocks.0.attn.rotary_sin', 'blocks.0.attn.rotary_cos', 'blocks.0.mlp.W_in', 'blocks.0.mlp.W_out', 'blocks.0.mlp.W_gate', 'blocks.0.mlp.b_in', 'blocks.0.mlp.b_out', 'blocks.1.attn.W_Q', 'blocks.1.attn.W_O', 'blocks.1.attn.b_Q', 'blocks.1.attn.b_O', 'blocks.1.attn.W_K', 'blocks.1.attn.W_V', 'blocks.1.attn.b_K', 'blocks.1.attn.b_V', 'blocks.1.attn.mask', 'blocks.1.attn.IGNORE', 'blocks.1.attn.rotary_sin', 'blocks.1.attn.rotary_cos', 'blocks.1.mlp.W_in', 'blocks.1.mlp.W_out', 'blocks.1.mlp.W_gate', 'blocks.1.mlp.b_in', 'blocks.1.mlp.b_out', 'blocks.2.attn.W_Q', 'blocks.2.attn.W_O', 'blocks.2.attn.b_Q', 'blocks.2.attn.b_O', 'blocks.2.attn.W_K', 'blocks.2.attn.W_V', 'blocks.2.attn.b_K', 'blocks.2.attn.b_V', 'blocks.2.attn.mask', 'blocks.2.attn.IGNORE', 'bl

In [70]:
data['category_stats'][(1,1,1)]

tensor([ 1,  1,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  2,  1, 15, 11,  1, 11,  1,  1,  1])